In [1]:
# Quantile model goal: train a model to identify 3 quantity breaks (low, medium, high) for each product.
# This is the unfinished version of the Quantile model.
# However, it had better Mean Absolute Error (MAE), Root Mean Squared Error (RMSE), and R^2 Scores than the combined model.

from pandas import read_csv
from pandas.plotting import scatter_matrix
from matplotlib import pyplot as plt
import seaborn as sns
import numpy as np
from scipy import stats
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.multioutput import MultiOutputRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score


# define dataset
dataset = read_csv("Intuilize_SalesData_Comp1_QB.csv")

# ensure times are datetime and calculate recency
dataset['SalesDate'] = pd.to_datetime(dataset['SalesDate'])
most_recent = dataset['SalesDate'].max()
dataset['Recency'] = (most_recent - dataset['SalesDate']).dt.days

# focus on data from the last 365 days
within_time_frame = dataset.loc[dataset['Recency'] <= 365].copy()

# create 3 buckets for each product
within_time_frame['small_quantile'] = within_time_frame.groupby('ProductID')['SalesQty'].transform(lambda x: x.quantile(0.33))
within_time_frame['medium_quantile'] = within_time_frame.groupby('ProductID')['SalesQty'].transform(lambda x: x.quantile(0.66))
within_time_frame['large_quantile'] = within_time_frame.groupby('ProductID')['SalesQty'].transform(lambda x: x.max())

# Split data into training and testing sets
X = within_time_frame[['SalesQty', 'UnitPrice']]
y = within_time_frame[['small_quantile','medium_quantile','large_quantile']]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Train regressor model
rf_model = RandomForestRegressor(n_estimators=100, random_state=42)
multi_rf_model = MultiOutputRegressor(rf_model)
multi_rf_model.fit(X_train, y_train)

# Predict and Evaluate for Each Quantity Break
y_pred = multi_rf_model.predict(X_test)

# Calculate evaluation metrics for each quantity break target
for i, target in enumerate(['small_quantile', 'medium_quantile', 'large_quantile']):
    print(f"\nEvaluation for {target}:")
    print("Mean Absolute Error (MAE):", mean_absolute_error(y_test[target], y_pred[:, i]))
    print("Mean Squared Error (MSE):", mean_squared_error(y_test[target], y_pred[:, i]))
    print("Root Mean Squared Error (RMSE):", np.sqrt(mean_squared_error(y_test[target], y_pred[:, i])))
    print("R^2 Score:", r2_score(y_test[target], y_pred[:, i]))


Evaluation for small_quantile:
Mean Absolute Error (MAE): 64.96057284992295
Mean Squared Error (MSE): 810029.0668995101
Root Mean Squared Error (RMSE): 900.0161481326377
R^2 Score: 0.9783350618311888

Evaluation for medium_quantile:
Mean Absolute Error (MAE): 99.31296573265429
Mean Squared Error (MSE): 1264162.4032407154
Root Mean Squared Error (RMSE): 1124.3497690846543
R^2 Score: 0.9754138231144607

Evaluation for large_quantile:
Mean Absolute Error (MAE): 444.9391590250255
Mean Squared Error (MSE): 20231279.945818104
Root Mean Squared Error (RMSE): 4497.91951304357
R^2 Score: 0.9688083021612586
